# 🛡️ Suraksha - Data Ingestion (Baseline Solution)

## Objective
Load **official hackathon dataset** for pattern-based fraud detection.

## Baseline Solution Strategy
* **Data Source**: Official CSV (`official_dataset.csv`)
* **Fraud Detection**: Pattern-based ONLY (no user tracking)
* **Key Difference**: Works WITHOUT user identifiers (VPAs, device IDs)
* **Target Table**: `workspace.bronze.upi_official`

## Why Baseline Solution?
> "Real banks have user identifiers, but official datasets rarely do. This proves our system adapts to data constraints while still detecting fraud through transaction patterns."

## Official Dataset Characteristics
* **17 columns** (transaction-level only)
* ❌ No sender/receiver VPAs
* ❌ No device IDs or SIM swap flags
* ❌ No behavioral velocity tracking
* ✅ Transaction patterns (amount, timing, merchant, device type)
* ✅ Binary fraud label

## Fraud Detection Approach
Detects **4 pattern-based fraud types**:
1. **Amount Anomaly** - Statistical outliers
2. **Temporal Anomaly** - Odd hour/weekend patterns
3. **Merchant Fraud** - High-risk merchant categories
4. **High-Risk Pattern** - Multiple suspicious indicators

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import os
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("🛡️ SURAKSHA - BASELINE DATA INGESTION")
print("="*70)

# Get current user dynamically
try:
    username = spark.sql("SELECT current_user()").collect()[0][0]
except:
    username = os.environ.get('USER', 'default_user')

print(f"\nCurrent user: {username}")

# Configuration
official_csv_path = f'/Workspace/Users/{username}/Suraksha/data/official_dataset.csv'
target_table = "workspace.bronze.upi_official"

print(f"\n📊 Configuration:")
print(f"   Source: {official_csv_path}")
print(f"   Target: {target_table}")
print(f"\n✓ Configuration loaded")

In [0]:
print("\n📊 Step 1: Loading official dataset from CSV...")

# Check if file exists
if not os.path.exists(official_csv_path):
    raise FileNotFoundError(f"Official dataset not found at: {official_csv_path}")

print(f"   File found: {official_csv_path}")
file_size_mb = os.path.getsize(official_csv_path) / (1024 * 1024)
print(f"   File size: {file_size_mb:.2f} MB")

# Load with pandas first (faster for inspection)
print("\n   Loading CSV with pandas...")
df_pandas = pd.read_csv(official_csv_path)

print(f"✓ Loaded {len(df_pandas):,} transactions")
print(f"✓ Columns: {len(df_pandas.columns)}")

# Show column list
print(f"\n📋 Column List:")
for i, col in enumerate(df_pandas.columns, 1):
    print(f"   {i:2d}. {col}")

# Check for user identifiers (should be NONE)
user_id_columns = ['sender_vpa', 'receiver_vpa', 'sender_id', 'receiver_id', 'device_id']
found_user_ids = [col for col in user_id_columns if col in df_pandas.columns]

if found_user_ids:
    print(f"\n⚠️  WARNING: Found user identifiers: {found_user_ids}")
    print("   This should be transaction-level data only!")
else:
    print(f"\n✓ Confirmed: No user identifiers (VPAs, device IDs)")
    print("   Perfect for pattern-based fraud detection!")

In [0]:
print("\n📊 Step 2: Running data quality checks...\n")

# 1. Check for nulls
print("1. Checking for missing values:")
null_counts = df_pandas.isnull().sum()
if null_counts.sum() > 0:
    print(f"   ⚠️  Found missing values:")
    for col, count in null_counts[null_counts > 0].items():
        print(f"      - {col}: {count:,} ({count/len(df_pandas)*100:.2f}%)")
else:
    print(f"   ✓ No missing values")

# 2. Fraud distribution
print(f"\n2. Fraud distribution:")
fraud_col = 'fraud_flag' if 'fraud_flag' in df_pandas.columns else 'is_fraud'
fraud_dist = df_pandas[fraud_col].value_counts()
legit_count = fraud_dist.get(0, 0)
fraud_count = fraud_dist.get(1, 0)
total = len(df_pandas)

print(f"   Legitimate: {legit_count:,} ({legit_count/total*100:.2f}%)")
print(f"   Fraud: {fraud_count:,} ({fraud_count/total*100:.2f}%)")

# 3. Date range
print(f"\n3. Date range:")
df_pandas['timestamp'] = pd.to_datetime(df_pandas['timestamp'])
print(f"   From: {df_pandas['timestamp'].min()}")
print(f"   To: {df_pandas['timestamp'].max()}")
date_range_days = (df_pandas['timestamp'].max() - df_pandas['timestamp'].min()).days
print(f"   Duration: {date_range_days} days")

# 4. Amount statistics
print(f"\n4. Amount statistics:")
amount_col = 'amount (INR)' if 'amount (INR)' in df_pandas.columns else 'amount_inr'
print(f"   Min: ₹{df_pandas[amount_col].min():,.2f}")
print(f"   Max: ₹{df_pandas[amount_col].max():,.2f}")
print(f"   Mean: ₹{df_pandas[amount_col].mean():,.2f}")
print(f"   Median: ₹{df_pandas[amount_col].median():,.2f}")

print(f"\n✅ Data quality checks complete!")

# 5. Display sample
print(f"\n🔍 Sample transactions:")
display(df_pandas.head(5))

In [0]:
print("\n📊 Step 3: Standardizing schema and converting to Spark...")

# Standardize column names (official dataset uses different naming)
column_mapping = {
    'transaction id': 'txn_id',
    'transaction type': 'txn_type',
    'amount (INR)': 'amount_inr',
    'transaction_status': 'txn_status',
    'fraud_flag': 'is_fraud'
}

# Rename columns
df_standardized = df_pandas.rename(columns=column_mapping)

print(f"✓ Standardized column names")
print(f"   Renamed {len(column_mapping)} columns")

# Convert to Spark DataFrame
print(f"\n   Converting to Spark DataFrame...")
spark_df = spark.createDataFrame(df_standardized)

print(f"✓ Spark DataFrame created")
print(f"   Rows: {spark_df.count():,}")
print(f"   Columns: {len(spark_df.columns)}")

print(f"\n📋 Schema:")
spark_df.printSchema()

In [0]:
print("\n📊 Step 4: Saving to Delta Lake bronze layer...")
print(f"   Target table: {target_table}")
print(f"   Format: Delta Lake with schema enforcement\n")

# Create catalog and schema if needed
spark.sql("CREATE CATALOG IF NOT EXISTS workspace")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")
print(f"✓ Unity Catalog structure ready")

# Write to Delta table
try:
    spark_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(target_table)
    
    print(f"✓ Data successfully written to Delta Lake!")
    print(f"   Table: {target_table}")
    print(f"   Rows: {spark_df.count():,}")
    
except Exception as e:
    print(f"❌ Error writing to Delta Lake: {str(e)}")
    raise

print(f"\n✅ Bronze layer ingestion complete!")

In [0]:
print("\n📊 Step 5: Verifying Delta Lake ingestion...\n")

# Read back from Delta table
verify_df = spark.read.table(target_table)

print(f"1. Row count verification:")
original_count = spark_df.count()
verified_count = verify_df.count()
print(f"   Original: {original_count:,}")
print(f"   Verified: {verified_count:,}")
if original_count == verified_count:
    print(f"   ✓ Row counts match!")
else:
    print(f"   ⚠️  Row count mismatch!")

print(f"\n2. Fraud distribution verification:")
fraud_dist_verify = verify_df.groupBy('is_fraud').count().orderBy('is_fraud').toPandas()
for _, row in fraud_dist_verify.iterrows():
    is_fraud = row['is_fraud']
    count = row['count']
    pct = count / verified_count * 100
    label = 'Fraud' if is_fraud == 1 else 'Legitimate'
    print(f"   {label}: {count:,} ({pct:.2f}%)")

print(f"\n3. Sample records from Delta table:")
display(verify_df.limit(10))

print(f"\n✅ Verification complete - data integrity confirmed!")

## ✓ Baseline Data Ingestion Summary

### What Was Loaded

**Data Source:** Official hackathon dataset (`official_dataset.csv`)  
**Target Table:** `workspace.bronze.upi_official`

**Key Characteristics:**
* ✅ **Transaction-level data only** (17 columns)
* ❌ **No user identifiers** (VPAs, device IDs, SIM swap flags)
* ❌ **No behavioral tracking** (velocity, history)
* ✅ **Pattern-based features** (amount, timing, merchant, device)

### Why This Matters

> **Dual Solution Strategy**: The advanced solution uses synthetic data with user identifiers to detect 9 fraud types. The baseline solution uses official data WITHOUT user tracking to prove adaptability and detect 4 pattern-based fraud types.

### Architecture Comparison

```
Advanced Solution                    Baseline Solution
├─ Synthetic data                   ├─ Official dataset
├─ WITH user identifiers            ├─ WITHOUT user identifiers
├─ 9 fraud types                    ├─ 4 pattern-based fraud types
├─ workspace.bronze.upi             ├─ workspace.bronze.upi_official
└─ Behavior + Pattern features     └─ Pattern features ONLY
```

### Next Steps

1. **02_pattern_features**: Engineer pattern-based features  
   - Amount statistical features (z-score, percentiles)
   - Temporal features (odd hours, weekends)
   - Merchant risk scoring
   - Transaction pattern combinations

2. **03_binary_training**: Train binary classifier  
   - XGBoost binary classification (Fraud vs Legitimate)
   - Pattern-based features only
   - Register as `suraksha_baseline` in MLflow

3. **04_model_serving**: Deploy baseline model  
   - Real-time inference API
   - Integrate with RAG for explanations
   - Side-by-side comparison with advanced model